# Coastal flood step 14: mangrove efficiency maps and rankings (USD per ha)

This notebook reuses outputs from step 09 to compute and visualize **mangrove protection efficiency**:

- `Net_Avoided_Efficiency_USD_per_ha = Avoided_EAD_USD / MangroveArea_ha`

Outputs include:

- parish and catchment efficiency maps (minimum + maximum scenarios)
- ranked charts (most protective and most negative areas)
- efficiency comparison tables (minimum vs maximum)


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300

jamaica_metric_grid_crs = 'EPSG:3448'

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')

geo_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/geographical_mangrove_attribution'
out_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/mangrove_efficiency_per_ha'
out_dir.mkdir(parents=True, exist_ok=True)

parish_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jam_adm_shp/jam_admbnda_adm1.shp'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
catchments_path = base_path / 'dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg'

print('Input summaries:', geo_dir)
print('Output folder:', out_dir)


In [ ]:
def load_efficiency_table(csv_path: Path, id_col: str) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f'Missing summary file: {csv_path}')

    df = pd.read_csv(csv_path)
    required = [id_col, 'MangroveArea_ha', 'Avoided_EAD_USD']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'Missing columns in {csv_path.name}: {missing}')

    for c in ['MangroveArea_ha', 'Avoided_EAD_USD', 'Positive_Avoided_EAD_USD', 'Negative_Avoided_EAD_USD', 'Area_ReducesDamage_ha', 'Area_IncreasesDamage_ha']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)

    # Net efficiency (can be positive or negative). NaN where mangrove area is zero.
    df['Net_Avoided_Efficiency_USD_per_ha'] = np.where(
        df['MangroveArea_ha'] > 0,
        df['Avoided_EAD_USD'] / df['MangroveArea_ha'],
        np.nan,
    )

    # Optional extra diagnostics for interpretation.
    if 'Positive_Avoided_EAD_USD' in df.columns and 'Area_ReducesDamage_ha' in df.columns:
        df['Positive_Efficiency_USD_per_ha'] = np.where(
            df['Area_ReducesDamage_ha'] > 0,
            df['Positive_Avoided_EAD_USD'] / df['Area_ReducesDamage_ha'],
            np.nan,
        )

    if 'Negative_Avoided_EAD_USD' in df.columns and 'Area_IncreasesDamage_ha' in df.columns:
        df['Negative_Efficiency_USD_per_ha'] = np.where(
            df['Area_IncreasesDamage_ha'] > 0,
            df['Negative_Avoided_EAD_USD'] / df['Area_IncreasesDamage_ha'],
            np.nan,
        )

    return df


parish_tables = {
    'minimum': load_efficiency_table(geo_dir / 'parish_summary_minimum.csv', 'ParishName'),
    'maximum': load_efficiency_table(geo_dir / 'parish_summary_maximum.csv', 'ParishName'),
}

catchment_tables = {
    'minimum': load_efficiency_table(geo_dir / 'catchment_summary_minimum.csv', 'catchment_uid'),
    'maximum': load_efficiency_table(geo_dir / 'catchment_summary_maximum.csv', 'catchment_uid'),
}

for scenario in ['minimum', 'maximum']:
    p = parish_tables[scenario]
    c = catchment_tables[scenario]
    print(f"{scenario} parish rows: {len(p)}, catchment rows: {len(c)}")


In [ ]:
parishes = gpd.read_file(parish_path).to_crs(jamaica_metric_grid_crs)
parishes = parishes[['NAME_1', 'geometry']].rename(columns={'NAME_1': 'ParishName'})

catchments = gpd.read_file(catchments_path).to_crs(jamaica_metric_grid_crs)
catchments = catchments[['catchment_uid', 'geometry']].copy()
catchments['catchment_uid'] = pd.to_numeric(catchments['catchment_uid'], errors='coerce')
catchments = catchments[catchments['catchment_uid'].notna()].copy()
catchments['catchment_uid'] = catchments['catchment_uid'].astype(int)

jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)

print('Parishes loaded:', len(parishes))
print('Catchments loaded:', len(catchments))


In [ ]:
def plot_efficiency_map(
    boundary_gdf: gpd.GeoDataFrame,
    summary_df: pd.DataFrame,
    key_col: str,
    value_col: str,
    scenario_name: str,
    geography_label: str,
    out_png: Path,
):
    map_df = boundary_gdf.merge(
        summary_df[[key_col, 'MangroveArea_ha', value_col]],
        on=key_col,
        how='left',
    )

    map_df['MangroveArea_ha'] = pd.to_numeric(map_df['MangroveArea_ha'], errors='coerce').fillna(0.0)
    map_df[value_col] = pd.to_numeric(map_df[value_col], errors='coerce')

    has_mangroves = map_df['MangroveArea_ha'] > 0
    vals = map_df.loc[has_mangroves, value_col].dropna()

    if len(vals) > 0:
        cap = float(vals.abs().quantile(0.98))
        if cap <= 0:
            cap = float(max(vals.abs().max(), 1.0))
    else:
        cap = 1.0

    fig, ax = plt.subplots(1, 1, figsize=(9.2, 8.3))

    # Draw no-mangrove areas in pale gray so efficiency is only mapped where relevant.
    if (~has_mangroves).any():
        map_df.loc[~has_mangroves].plot(
            ax=ax,
            color='#EFEFEF',
            edgecolor='#D0D0D0',
            linewidth=0.2,
            zorder=1,
        )

    if has_mangroves.any():
        map_df.loc[has_mangroves].plot(
            ax=ax,
            column=value_col,
            cmap='RdYlGn',
            norm=TwoSlopeNorm(vmin=-cap, vcenter=0.0, vmax=cap),
            edgecolor='#6F6F6F',
            linewidth=0.2,
            legend=True,
            legend_kwds={'label': 'Net avoided EAD efficiency (USD per mangrove ha)'},
            zorder=2,
        )

    jamaica_boundary.boundary.plot(ax=ax, color='black', linewidth=0.45, zorder=3)

    ax.set_title(f"{geography_label} mangrove efficiency - {scenario_name}\n(USD avoided EAD per ha)", fontsize=12)
    ax.set_axis_off()
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', out_png)

In [ ]:
for scenario_name, df in parish_tables.items():
    out_png = out_dir / f'parish_efficiency_usd_per_ha_{scenario_name}.png'
    plot_efficiency_map(
        boundary_gdf=parishes,
        summary_df=df,
        key_col='ParishName',
        value_col='Net_Avoided_Efficiency_USD_per_ha',
        scenario_name=scenario_name,
        geography_label='Parish',
        out_png=out_png,
    )

for scenario_name, df in catchment_tables.items():
    out_png = out_dir / f'catchment_efficiency_usd_per_ha_{scenario_name}.png'
    plot_efficiency_map(
        boundary_gdf=catchments,
        summary_df=df,
        key_col='catchment_uid',
        value_col='Net_Avoided_Efficiency_USD_per_ha',
        scenario_name=scenario_name,
        geography_label='Catchment',
        out_png=out_png,
    )


In [ ]:
def make_rank_outputs(df: pd.DataFrame, id_col: str, scenario_name: str, geography_name: str, top_n: int = 12):
    t = df[[id_col, 'MangroveArea_ha', 'Avoided_EAD_USD', 'Net_Avoided_Efficiency_USD_per_ha']].copy()
    t = t[(t['MangroveArea_ha'] > 0) & (t['Net_Avoided_Efficiency_USD_per_ha'].notna())].copy()

    top_pos = t.sort_values('Net_Avoided_Efficiency_USD_per_ha', ascending=False).head(top_n).copy()
    top_neg = t.sort_values('Net_Avoided_Efficiency_USD_per_ha', ascending=True).head(top_n).copy()

    top_pos_out = out_dir / f'{geography_name.lower()}_top_positive_efficiency_{scenario_name}.csv'
    top_neg_out = out_dir / f'{geography_name.lower()}_top_negative_efficiency_{scenario_name}.csv'
    top_pos.to_csv(top_pos_out, index=False)
    top_neg.to_csv(top_neg_out, index=False)

    # One ranking chart including strongest positive and negative areas.
    rank_plot = pd.concat([
        top_neg.assign(_group='Most negative'),
        top_pos.assign(_group='Most protective'),
    ], ignore_index=True)
    rank_plot = rank_plot.sort_values('Net_Avoided_Efficiency_USD_per_ha', ascending=True)

    fig, ax = plt.subplots(1, 1, figsize=(11, 8))
    colors = np.where(
        rank_plot['Net_Avoided_Efficiency_USD_per_ha'] >= 0,
        '#2E7D32',
        '#B71C1C',
    )

    ax.barh(
        rank_plot[id_col].astype(str),
        rank_plot['Net_Avoided_Efficiency_USD_per_ha'],
        color=colors,
        alpha=0.9,
    )
    ax.axvline(0, color='black', linewidth=1.0)
    ax.set_xlabel('Net avoided EAD efficiency (USD per mangrove ha)')
    ax.set_ylabel(geography_name)
    ax.set_title(f'{geography_name} efficiency ranking - {scenario_name}')
    ax.grid(axis='x', alpha=0.25, linestyle='--')

    out_png = out_dir / f'{geography_name.lower()}_efficiency_ranking_{scenario_name}.png'
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', out_png)
    print('Saved:', top_pos_out)
    print('Saved:', top_neg_out)


for scenario_name, df in parish_tables.items():
    make_rank_outputs(df, 'ParishName', scenario_name, 'Parish', top_n=10)

for scenario_name, df in catchment_tables.items():
    make_rank_outputs(df, 'catchment_uid', scenario_name, 'Catchment', top_n=12)


In [ ]:
def build_efficiency_compare(min_df: pd.DataFrame, max_df: pd.DataFrame, key_col: str):
    comp = (
        min_df[[key_col, 'Net_Avoided_Efficiency_USD_per_ha']]
        .rename(columns={'Net_Avoided_Efficiency_USD_per_ha': 'Min_Net_Avoided_Efficiency_USD_per_ha'})
        .merge(
            max_df[[key_col, 'Net_Avoided_Efficiency_USD_per_ha']].rename(
                columns={'Net_Avoided_Efficiency_USD_per_ha': 'Max_Net_Avoided_Efficiency_USD_per_ha'}
            ),
            on=key_col,
            how='outer',
        )
    )

    comp['Delta_Efficiency_USD_per_ha_MaxMinusMin'] = (
        comp['Max_Net_Avoided_Efficiency_USD_per_ha'] - comp['Min_Net_Avoided_Efficiency_USD_per_ha']
    )
    comp = comp.sort_values('Delta_Efficiency_USD_per_ha_MaxMinusMin', ascending=False).reset_index(drop=True)
    return comp


parish_compare = build_efficiency_compare(parish_tables['minimum'], parish_tables['maximum'], 'ParishName')
catchment_compare = build_efficiency_compare(catchment_tables['minimum'], catchment_tables['maximum'], 'catchment_uid')

parish_compare_out = out_dir / 'parish_efficiency_compare_minimum_vs_maximum.csv'
catchment_compare_out = out_dir / 'catchment_efficiency_compare_minimum_vs_maximum.csv'

parish_compare.to_csv(parish_compare_out, index=False)
catchment_compare.to_csv(catchment_compare_out, index=False)

print('Saved:', parish_compare_out)
print('Saved:', catchment_compare_out)

display(parish_compare.head(10))
display(catchment_compare.head(12))


In [ ]:
for scenario_name, df in parish_tables.items():
    df.to_csv(out_dir / f'parish_efficiency_usd_per_ha_{scenario_name}.csv', index=False)

for scenario_name, df in catchment_tables.items():
    df.to_csv(out_dir / f'catchment_efficiency_usd_per_ha_{scenario_name}.csv', index=False)

print('Saved efficiency tables and visual outputs in:', out_dir)
